In [75]:
import sympy as sp
import numpy as np
import roboticstoolbox as rtp

th1, th2, th3, th4, th5 = sp.symbols('th1 th2 th3 th4 th5', real=True)
# السرعات الزاوية (Angular Velocities)
dth1, dth2, dth3, dth4, dth5 = sp.symbols('dth1 dth2 dth3 dth4 dth5', real=True)

# التسارعات الزاوية (Angular Accelerations)
ddth1, ddth2, ddth3, ddth4, ddth5 = sp.symbols('ddth1 ddth2 ddth3 ddth4 ddth5', real=True)
# ==============================
# DH symbolic transform
# ==============================





def DH(a, alpha, d, th):
    return sp.Matrix([
        [sp.cos(th), -sp.sin(th)*sp.cos(alpha),  sp.sin(th)*sp.sin(alpha), a*sp.cos(th)],
        [sp.sin(th),  sp.cos(th)*sp.cos(alpha), -sp.cos(th)*sp.sin(alpha), a*sp.sin(th)],
        [0,            sp.sin(alpha),            sp.cos(alpha),             d          ],
        [0,            0,                        0,                          1          ]
    ])

# ==============================
# LINKS A1..A5
# ==============================
T1 = DH(4.7, sp.pi/2,5, th1)#cm
T2 = DH(12,  0,       0,th2+sp.pi/2)#cm
T3 = DH(4.4, sp.pi/2, 0,th3)#cm
T4 = DH(0, sp.pi/2, 15, th4)#cm
T5 = DH(8, 0,       0,    th5+sp.pi/2)#cm

# ==============================
# Q MATRIX
# ==============================
Q = sp.Matrix([
    [0, -1, 0, 0],
    [1,  0, 0, 0],
    [0,  0, 0, 0],
    [0,  0, 0, 0]
])

# ==============================
# Uij — manual
# ==============================
U11 = Q*T1;                  U12 = sp.zeros(4);     U13 = sp.zeros(4);     U14 = sp.zeros(4);     U15 = sp.zeros(4)
U21 = Q*T1*T2;               U22 = T1*Q*T2;         U23 = sp.zeros(4);     U24 = sp.zeros(4);     U25 = sp.zeros(4)
U31 = Q*T1*T2*T3;            U32 = T1*Q*T2*T3;      U33 = T1*T2*Q*T3;      U34 = sp.zeros(4);     U35 = sp.zeros(4)
U41 = Q*T1*T2*T3*T4;         U42 = T1*Q*T2*T3*T4;   U43 = T1*T2*Q*T3*T4;   U44 = T1*T2*T3*Q*T4;   U45 = sp.zeros(4)
U51 = Q*T1*T2*T3*T4*T5;      U52 = T1*Q*T2*T3*T4*T5;U53 = T1*T2*Q*T3*T4*T5;U54 = T1*T2*T3*Q*T4*T5;U55 = T1*T2*T3*T4*Q*T5

U = [
    [U11,U12,U13,U14,U15],
    [U21,U22,U23,U24,U25],
    [U31,U32,U33,U34,U35],
    [U41,U42,U43,U44,U45],
    [U51,U52,U53,U54,U55]
]

# ==============================
# INERTIA MATRICES J1..J5
# ==============================
J1 = sp.Matrix([[2211,0,0,-75],[0,3751,0,-67],[0,0,1940,404.5],[-75,-67,404.5,257.7]])
J2 = sp.Matrix([[377.88,0,0,0],[0,5632,0,690.8],[0,0,884.5,47.18],[0,690.5,47.18,168.5]])
J3 = sp.Matrix([[661.47,0,0,334],[0,670,0,50.5],[0,0,377.5,-19],[334,50.5,-19,90]])
J4 = sp.Matrix([[295,0,0,38.6],[0,1303.2,0,-3.86],[0,0,8235,1323.9],[38.6,-3.86,1323.9,193]])
J5 = sp.Matrix([[300,0,0,13.875],[0,2751,0,425.725],[0,0,376,-297],[13.875,425.7,-297,92.5]])
J = [J1,J2,J3,J4,J5]


# --- LINK 1 ---
m1 = 257.7#grams
rc1 = sp.Matrix([-0.29, -0.26,1.57])#cm


# --- LINK 2 ---
m2 = 168.5#grams
rc2 = sp.Matrix([0,4.1,0.28])#cm



# --- LINK 3 ---
m3 = 90#grams
rc3 = sp.Matrix([3.71,0.56,-0.21])#cm


# --- LINK 4 ---
m4 =193#grams
rc4 = sp.Matrix([0.2,-0.02,6.86])#cm


# --- LINK 5 ---
m5 =92.5#grams
rc5 = sp.Matrix([0.15,4.6,-3.21])#cm


#حساب Dij
#Dij=Dji

# [ D11   D12   D13   D14   D15 ]
# [ ---   D22   D23   D24   D25 ]
# [ ---   ---   D33   D34   D35 ]
# [ ---   ---   ---   D44   D45 ]
# [ ---   ---   ---   ---   D55 ]

Iact1 = 4e-6 * 1e7   # = 40  g·cm^2
Iact2 = 1e-8 * 1e7   # = 0.1 g·cm^2


#D11 = Trace(U11*J1*U11') + Trace(U21*J2*U21') + Trace(U31*J3*U31') + Trace(U41*J4*U41') + Trace(U51*J5*U51')
D11= sp.trace(U11*J1*U11.T) + sp.trace(U21*J2*U21.T) + sp.trace(U31*J3*U31.T) + sp.trace(U41*J4*U41.T) + sp.trace(U51*J5*U51.T)
D11=sp.expand(D11)+Iact1


#D22 = Trace(U22*J2*U22') + Trace(U32*J3*U32') + Trace(U42*J4*U42') + Trace(U52*J5*U52')
D22= sp.trace(U22*J2*U22.T) + sp.trace(U32*J3*U32.T) + sp.trace(U42*J4*U42.T) + sp.trace(U52*J5*U52.T)
D22=sp.expand(D22)+Iact1

#D33 = Trace(U33*J3*U33') + Trace(U43*J4*U43') + Trace(U53*J5*U53')
D33= sp.trace(U33*J3*U33.T) + sp.trace(U43*J4*U43.T) + sp.trace(U53*J5*U53.T)
D33=sp.expand(D33)+Iact1

#D44 = Trace(U44*J4*U44') + Trace(U54*J5*U54')
D44= sp.trace(U44*J4*U44.T) + sp.trace(U54*J5*U54.T)
D44=sp.expand(D44)+Iact1

#D55 = Trace(U55*J5*U55')
D55= sp.trace(U55*J5*U55.T)
D55=sp.expand(D55)+Iact2



#D12 = Trace(U22*J2*U21') + Trace(U32*J3*U31') + Trace(U42*J4*U41') + Trace(U52*J5*U51')
#D13 = Trace(U33*J3*U31') + Trace(U43*J4*U41') + Trace(U53*J5*U51')
#D14 = Trace(U44*J4*U41') + Trace(U54*J5*U51')
#D15 = Trace(U55*J5*U51')
D12= sp.trace(U22*J2*U21.T) + sp.trace(U32*J3*U31.T) + sp.trace(U42*J4*U41.T) + sp.trace(U52*J5*U51.T)
D12=sp.expand(D12)
D21=D12

D13= sp.trace(U33*J3*U31.T) + sp.trace(U43*J4*U41.T) + sp.trace(U53*J5*U51.T)
D13=sp.expand(D13)
D31=D13

D14= sp.trace(U44*J4*U41.T) + sp.trace(U54*J5*U51.T)
D14=sp.expand(D14)
D41=D14


D15= sp.trace(U55*J5*U51.T)
D15=sp.expand(D15)

D51=D15

#D23 = Trace(U33*J3*U32') + Trace(U43*J4*U42') + Trace(U53*J5*U52')
#D24 = Trace(U44*J4*U42') + Trace(U54*J5*U52')
#D25 = Trace(U55*J5*U52')

D23= sp.trace(U33*J3*U32.T) + sp.trace(U43*J4*U42.T) + sp.trace(U53*J5*U52.T)
D23=sp.expand(D23)
D32=D23


D24= sp.trace(U44*J4*U42.T) + sp.trace(U54*J5*U52.T)
D24=sp.expand(D24)
D42=D24

D25= sp.trace(U55*J5*U52.T)
D25=sp.expand(D25)
D52=D25

#D34 = Trace(U44*J4*U43') + Trace(U54*J5*U53')
#D35 = Trace(U55*J5*U53')

D34= sp.trace(U44*J4*U43.T) + sp.trace(U54*J5*U53.T)
D34=sp.expand(D34)
D43=D34

D35= sp.trace(U55*J5*U53.T)
D35=sp.expand(D35)
D53=D35

#D45 = Trace(U55*J5*U54')
D45= sp.trace(U55*J5*U54.T)
D45=sp.expand(D45)
D54=D45




#Di




r1 = sp.Matrix([-0.29, -0.26,1.57, 1])#cm


r2 = sp.Matrix([0,4.1,0.28, 1])#cm


r3 = sp.Matrix([3.71,0.56,-0.21, 1])#cm


r4 = sp.Matrix([0.2,-0.02,6.86, 1])#cm

r5 = sp.Matrix([0.15,4.6,-3.21, 1])#cm

g1=sp.Matrix([0,0,-981,0])#cm/s^2
g2=sp.Matrix([0,-981,0,0])
g3=sp.Matrix([-981,0,0,0])
g4=sp.Matrix([-981,0,0,0])
g5=sp.Matrix([981,0,0,0])



# D1: Gravity torque for Joint 1
# D1 = -m1*(g1.T*U11*r1) - m2*(g2.T*U21*r2) - m3*(g3.T*U31*r3) - m4*(g4.T*U41*r4) - m5*(g5.T*U51*r5)
D1=-m1*(g1.T*U11*r1)-m2*(g2.T*U21*r2)-m3*(g3.T*U31*r3)-m4*(g4.T*U41*r4)-m5*(g5.T*U51*r5)
D1=sp.expand(D1[0])

# D2 (Gravity Torque at Joint 2):
# D2 =  -m2*(g2.T*U22*r2) - m3*(g3.T*U32*r3) - m4*(g4.T*U42*r4) - m5*(g5.T*U52*r5)
D2=-m2*(g2.T*U22*r2)-m3*(g3.T*U32*r3)-m4*(g4.T*U42*r4)-m5*(g5.T*U52*r5)
D2=sp.expand(D2[0])

# D3 (Gravity Torque at Joint 3):
# D3 = -m3*(g3.T*U33*r3) - m4*(g4.T*U43*r4) - m5*(g5.T*U53*r5) 
D3=-m3*(g3.T*U33*r3)-m4*(g4.T*U43*r4)-m5*(g5.T*U53*r5)
D3=sp.expand(D3[0])



# D4 (Gravity Torque at Joint 4):
# D4 =  -m4*(g4.T*U44*r4) - m5*(g5.T*U54*r5)
D4=-m4*(g4.T*U44*r4)-m5*(g5.T*U54*r5)
D4=sp.expand(D4[0])



# D5 (Gravity Torque at Joint 5):
# D5 =  -m5*(g5.T*U55*r5)
D5=-m5*(g5.T*U55*r5)
D5=sp.expand(D5[0])







#المعادلات الديناميكيا للنموذج


Iact1 = 4e-6 * 1e7   # = 40  g·cm^2
Iact2 = 1e-8 * 1e7   # = 0.1 g·cm^2



tau1=(D11*ddth1+D12*ddth2+D13*ddth3+D14*ddth4+D15*ddth5+D1)*1e-7
tau2=(D21*ddth1+D22*ddth2+D23*ddth3+D24*ddth4+D25*ddth5+D2)*1e-7
tau3=(D31*ddth1+D32*ddth2+D33*ddth3+D34*ddth4+D35*ddth5+D3)*1e-7
tau4=(D41*ddth1+D42*ddth2+D43*ddth3+D44*ddth4+D45*ddth5+D4)*1e-7
tau5=(D51*ddth1+D52*ddth2+D53*ddth3+D54*ddth4+D55*ddth5+D5)*1e-7

print(f"tau1={sp.expand(tau1)}\n")
print(f"tau2={tau2}\n")
print(f"tau3={tau3}\n")
print(f"tau4={tau4}\n")
print(f"tau5={tau5}\n")





subs_dict = {
    th1: 0, th2: 0, th3: 0, th4: 0, th5: 0,
    ddth1: 0, ddth2: 0, ddth3: 0, ddth4: 0, ddth5: 0
}

tau1_num = sp.N(tau1.subs(subs_dict))
tau2_num = sp.N(tau2.subs(subs_dict))
tau3_num = sp.N(tau3.subs(subs_dict))
tau4_num = sp.N(tau4.subs(subs_dict))
tau5_num = sp.N(tau5.subs(subs_dict))

print(f"tau1 = {tau1_num} m.N")
print(f"tau2 = {tau2_num} m.N")
print(f"tau3 ={tau3_num} m.N")
print(f"tau4 = {tau4_num} m.N")
print(f"tau5 = {tau5_num} m.N")





tau1=0.0002751*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2*sin(th5)**2 - 0.00068114*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2*sin(th5)*cos(th5) - 0.0012771375*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2*sin(th5) + 0.0006442*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2*cos(th5)**2 + 0.002261625*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2*cos(th5) + 0.00658024*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)**2 - 0.0004752*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th4)*cos(th3)*cos(th5) + 0.0030807*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th4)*cos(th3) + 0.00068114*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th5)**2*cos(th3)*cos(th4) - 0.0007382*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th5)*cos(th3)*cos(th4)*cos(th5) - 0.002261625*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th5)*cos(th3)*cos(th4) - 0.000374627*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th5)*cos(th3) - 0.00102171*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*sin(th5) - 0.00068114*ddth1*sin(th1)**2*sin(th2)**2*sin(th3)*cos(th3)*cos(th4)*cos(th5)

In [76]:
#MG995
Ra=2.5 #ome
K=0.02 #N.M/A
N=200
JJ=3*1e-6 #kg.m^2
B=2*1e-6 #N.m.s/rad
#SG90
Ra1=8.5 #ome
N1=120
K1=0.01 #N.M/A
JJ1=1*1e-6 #kg.m^2
B1=1*1e-6 #N.m.s/rad
b=0.25 #احتكاك


In [77]:
U1,U2,U3,U4,U5=sp.symbols('U1 U2 U3 U4 U5')

R1=(N**2*B+(K**2*N**2/Ra)+b)*dth1+D1
R2=(N**2*B+(K**2*N**2/Ra)+b)*dth2+D2
R3=(N**2*B+(K**2*N**2/Ra)+b)*dth3+D3
R4=(N**2*B+(K**2*N**2/Ra)+b)*dth4+D4
R5=(N1**2*B1+(K1**2*N1**2/Ra1)+b)*dth5+D5


theta_doble_dot=sp.Matrix([[ddth1],
                           [ddth2],
                           [ddth3],
                           [ddth4],
                           [ddth5]])


Dij=sp.Matrix([[D11+N**2*JJ, D12, D13, D14, D15],
                [D21, D22+N**2*JJ, D23, D24, D25],
                [D31, D32, D33+N**2*JJ, D34, D35],
                [D41, D42, D43, D44+N**2*JJ, D45],
                [D51, D52, D53, D54, D55+N**2*JJ1]])*1e-7 #Kg.m^2

#Mij=Dij.inv()


#theta_doble_dot=Mij*(sp.Matrix([[K*(N/Ra)*U1-R1],
      #                    [K*(N/Ra)*U2-R2],
         #                   [K*(N/Ra)*U3-R3],
         #                   [K*(N/Ra)*U4-R4],
          #                  [K1*(N1/Ra1)*U5-R5]]))
rhs=sp.Matrix([[K*(N/Ra)*U1-R1],
                          [K*(N/Ra)*U2-R2],
                            [K*(N/Ra)*U3-R3],
                            [K*(N/Ra)*U4-R4],
                            [K1*(N1/Ra1)*U5-R5]])

theta_doble_dot= Dij.LUsolve(rhs)



In [78]:
subs_q0 = {
    th1:0, th2:0, th3:0, th4:0, th5:0
}

Dij_0 = Dij.subs(subs_q0)
Mij_0 = Dij_0.inv()     
D1_0 = D1.subs(subs_q0)*1e-7#Kg.m^2
D2_0 = D2.subs(subs_q0)*1e-7
D3_0 = D3.subs(subs_q0)*1e-7
D4_0 = D4.subs(subs_q0)*1e-7
D5_0 = D5.subs(subs_q0)*1e-7

M11=Mij_0[0,0]
M12=Mij_0[0,1]
M13=Mij_0[0,2]
M14=Mij_0[0,3]
M15=Mij_0[0,4]
M21=Mij_0[1,0]
M22=Mij_0[1,1]
M23=Mij_0[1,2]
M24=Mij_0[1,3]
M25=Mij_0[1,4]
M31=Mij_0[2,0]
M32=Mij_0[2,1]
M33=Mij_0[2,2]
M34=Mij_0[2,3]
M35=Mij_0[2,4]
M41=Mij_0[3,0]
M42=Mij_0[3,1]
M43=Mij_0[3,2]
M44=Mij_0[3,3]
M45=Mij_0[3,4]
M51=Mij_0[4,0]
M52=Mij_0[4,1]
M53=Mij_0[4,2]
M54=Mij_0[4,3]
M55=Mij_0[4,4]

R1=(N**2*B+(K**2*N**2/Ra)+b)*dth1+D1_0
R2=(N**2*B+(K**2*N**2/Ra)+b)*dth2+D2_0
R3=(N**2*B+(K**2*N**2/Ra)+b)*dth3+D3_0
R4=(N**2*B+(K**2*N**2/Ra)+b)*dth4+D4_0
R5=(N1**2*B1+(K1**2*N1**2/Ra1)+b)*dth5+D5_0
#متحولات الحالة
#x1=th1
#x2=dth1
#x3=th2
#x4=dth2
#x5=th3
#x6=dth3
#x7=th4
#x8=dth4
#x9=th5
#x10=dth5
x1,x2,x3,x4,x5,x6,x7,x8,x9,x10=sp.symbols('x1 x2 x3 x4 x5 x6 x7 x8 x9 x10')
dx1,dx2,dx3,dx4,dx5,dx6,dx7,dx8,dx9,dx10=sp.symbols('dx1 dx2 dx3 dx4 dx5 dx6 dx7 dx8 dx9 dx10')
x=sp.Matrix([[x1],[x2],[x3],[x4],[x5],[x6],[x7],[x8],[x9],[x10]])
dx=sp.Matrix([[dx1],[dx2],[dx3],[dx4],[dx5],[dx6],[dx7],[dx8],[dx9],[dx10]])
dx=sp.Matrix([[0,1,0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,1,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,1,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,1,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,1],
                [0,0,0,0,0,0,0,0,0,0]])*x+sp.Matrix([[0,0,0,0,0],
                                                     [M11,M12,M13,M14,M15],
                                                     [0,0,0,0,0],
                                                     [M21,M22,M23,M24,M25],
                                                     [0,0,0,0,0],
                                                     [M31,M32,M33,M34,M35],
                                                     [0,0,0,0,0],
                                                     [M41,M42,M43,M44,M45],
                                                     [0,0,0,0,0],
                                                     [M51,M52,M53,M54,M55]])*sp.Matrix([[K*(N/Ra)*U1-R1],
                                                                                       [K*(N/Ra)*U2-R2],
                                                                                       [K*(N/Ra)*U3-R3],
                                                                                       [K*(N/Ra)*U4-R4],
                                                                                       [K1*(N1/Ra1)*U5-R5]])
                                                     

dx_=dx.subs({th1:x1, dth1:x2, th2:x3, dth2:x4, th3:x5, dth3:x6, th4:x7, dth4:x8, th5:x9, dth5:x10})
dx_expanded = sp.expand(dx_)
for i in range(10):
    print(f"dx{i+1} = {dx_expanded[i,0]}\n")


dx1 = x2

dx2 = 108.821088863245*U1 - 45.6777332887414*U2 + 120.397707953629*U3 - 234.05085207745*U4 + 20.9363672437279*U5 - 64.3339671454352*x10 - 457.728705031023*x2 + 192.131965645768*x4 - 506.422859079951*x6 + 984.476396550774*x8 + 4.70742872701556

dx3 = x4

dx4 = -45.6777332887414*U1 + 306.190390020691*U2 - 570.648583563181*U3 + 507.682079801535*U4 - 73.8734743256199*U5 + 227.000874356909*x10 + 192.131965645768*x2 - 1287.91332802453*x4 + 2400.29060461263*x6 - 2135.43774816521*x8 + 34.1123230135278

dx5 = x6

dx6 = 120.397707953629*U1 - 570.648583563181*U2 + 1486.57920536266*U3 - 1662.33394474605*U4 + 229.648929645841*U5 - 705.672885980061*x10 - 506.422859079951*x2 + 2400.29060461263*x4 - 6252.92378255669*x6 + 6992.19215508807*x8 - 45.149769199401

dx7 = x8

dx8 = -234.05085207745*U1 + 507.682079801535*U2 - 1662.33394474605*U3 + 3595.91593191811*U4 - 313.337268330785*U5 + 962.833202702448*x10 + 984.476396550774*x2 - 2135.43774816521*x4 + 6992.19215508807*x6 - 15125.3213886305*x8 +

In [79]:
import sympy as sp

# ===============================
# ثوابت محركات MG995
# ===============================
Ra = 2.5       # Ω
K = 0.02       # N.m/A
N = 200
JJ = 3e-6      # kg.m^2
B = 2e-6       # N.m.s/rad

# ===============================
# ثوابت محركات SG90
# ===============================
Ra1 = 8.5      # Ω
N1 = 120
K1 = 0.01      # N.m/A
JJ1 = 1e-6     # kg.m^2
B1 = 1e-6      # N.m.s/rad

b = 0.25       # احتكاك

# ===============================
# رموز المتغيرات
# ===============================
U1, U2, U3, U4, U5 = sp.symbols('U1 U2 U3 U4 U5')
th1, th2, th3, th4, th5 = sp.symbols('th1 th2 th3 th4 th5', real=True)
dth1, dth2, dth3, dth4, dth5 = sp.symbols('dth1 dth2 dth3 dth4 dth5')
ddth1, ddth2, ddth3, ddth4, ddth5 = sp.symbols('ddth1 ddth2 ddth3 ddth4 ddth5')


# ===============================
# تعريف R
# ===============================
R1 = (N**2*B + (K**2 * N**2 / Ra) + b) * dth1 + D1
R2 = (N**2*B + (K**2 * N**2 / Ra) + b) * dth2 + D2
R3 = (N**2*B + (K**2 * N**2 / Ra) + b) * dth3 + D3
R4 = (N**2*B + (K**2 * N**2 / Ra) + b) * dth4 + D4
R5 = (N1**2*B1 + (K1**2 * N1**2 / Ra1) + b) * dth5 + D5

# ===============================
# مصفوفة Dij
# ===============================
Dij = sp.Matrix([
    [D11 + N**2*JJ, D12, D13, D14, D15],
    [D21, D22 + N**2*JJ, D23, D24, D25],
    [D31, D32, D33 + N**2*JJ, D34, D35],
    [D41, D42, D43, D44 + N**2*JJ, D45],
    [D51, D52, D53, D54, D55 + N**2*JJ1]
]) * 1e-7  # kg.m^2

# ===============================
# مصفوفة RHS
# ===============================
rhs = sp.Matrix([
    [K*(N/Ra)*U1 - R1],
    [K*(N/Ra)*U2 - R2],
    [K*(N/Ra)*U3 - R3],
    [K*(N/Ra)*U4 - R4],
    [K1*(N1/Ra1)*U5 - R5]
])

# ===============================
# حساب التسارع الزاوي
# ===============================
theta_doble_dot = Dij.LUsolve(rhs)

# ===============================
# التعويض عند وضعية صفرية
# ===============================
subs_q0 = {th1:0, th2:0, th3:0, th4:0, th5:0}
Dij_0 = Dij.subs(subs_q0)
Mij_0 = Dij_0.inv()
D1_0 = D1.subs(subs_q0)*1e-7
D2_0 = D2.subs(subs_q0)*1e-7
D3_0 = D3.subs(subs_q0)*1e-7
D4_0 = D4.subs(subs_q0)*1e-7
D5_0 = D5.subs(subs_q0)*1e-7

# ===============================
# استخراج عناصر المصفوفة Mij
# ===============================
M11, M12, M13, M14, M15 = Mij_0[0,0], Mij_0[0,1], Mij_0[0,2], Mij_0[0,3], Mij_0[0,4]
M21, M22, M23, M24, M25 = Mij_0[1,0], Mij_0[1,1], Mij_0[1,2], Mij_0[1,3], Mij_0[1,4]
M31, M32, M33, M34, M35 = Mij_0[2,0], Mij_0[2,1], Mij_0[2,2], Mij_0[2,3], Mij_0[2,4]
M41, M42, M43, M44, M45 = Mij_0[3,0], Mij_0[3,1], Mij_0[3,2], Mij_0[3,3], Mij_0[3,4]
M51, M52, M53, M54, M55 = Mij_0[4,0], Mij_0[4,1], Mij_0[4,2], Mij_0[4,3], Mij_0[4,4]

# ===============================
# إعادة تعريف R عند وضعية صفرية
# ===============================
R1 = (N**2*B + (K**2*N**2/Ra) + b) * dth1 + D1_0
R2 = (N**2*B + (K**2*N**2/Ra) + b) * dth2 + D2_0
R3 = (N**2*B + (K**2*N**2/Ra) + b) * dth3 + D3_0
R4 = (N**2*B + (K**2*N**2/Ra) + b) * dth4 + D4_0
R5 = (N1**2*B1 + (K1**2*N1**2/Ra1) + b) * dth5 + D5_0

# ===============================
# متحولات الحالة
# ===============================
x1,x2,x3,x4,x5,x6,x7,x8,x9,x10 = sp.symbols('x1 x2 x3 x4 x5 x6 x7 x8 x9 x10')
dx1,dx2,dx3,dx4,dx5,dx6,dx7,dx8,dx9,dx10 = sp.symbols('dx1 dx2 dx3 dx4 dx5 dx6 dx7 dx8 dx9 dx10')
x = sp.Matrix([[x1],[x2],[x3],[x4],[x5],[x6],[x7],[x8],[x9],[x10]])

dx = sp.Matrix([
    [0,1,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,1,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,1,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,1,0,0],
    [0,0,0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0,0,1],
    [0,0,0,0,0,0,0,0,0,0]
]) * x + sp.Matrix([
    [0,0,0,0,0],
    [M11,M12,M13,M14,M15],
    [0,0,0,0,0],
    [M21,M22,M23,M24,M25],
    [0,0,0,0,0],
    [M31,M32,M33,M34,M35],
    [0,0,0,0,0],
    [M41,M42,M43,M44,M45],
    [0,0,0,0,0],
    [M51,M52,M53,M54,M55]
]) * sp.Matrix([[K*(N/Ra)*U1 - R1],
                [K*(N/Ra)*U2 - R2],
                [K*(N/Ra)*U3 - R3],
                [K*(N/Ra)*U4 - R4],
                [K1*(N1/Ra1)*U5 - R5]])

# ===============================
# استبدال المتغيرات
# ===============================
dx_ = dx.subs({th1:x1, dth1:x2, th2:x3, dth2:x4, th3:x5, dth3:x6, th4:x7, dth4:x8, th5:x9, dth5:x10})
for i in range(10):
    print(f"dx{i+1} = {dx_expanded[i,0]}\n")



dx1 = x2

dx2 = 108.821088863245*U1 - 45.6777332887414*U2 + 120.397707953629*U3 - 234.05085207745*U4 + 20.9363672437279*U5 - 64.3339671454352*x10 - 457.728705031023*x2 + 192.131965645768*x4 - 506.422859079951*x6 + 984.476396550774*x8 + 4.70742872701556

dx3 = x4

dx4 = -45.6777332887414*U1 + 306.190390020691*U2 - 570.648583563181*U3 + 507.682079801535*U4 - 73.8734743256199*U5 + 227.000874356909*x10 + 192.131965645768*x2 - 1287.91332802453*x4 + 2400.29060461263*x6 - 2135.43774816521*x8 + 34.1123230135278

dx5 = x6

dx6 = 120.397707953629*U1 - 570.648583563181*U2 + 1486.57920536266*U3 - 1662.33394474605*U4 + 229.648929645841*U5 - 705.672885980061*x10 - 506.422859079951*x2 + 2400.29060461263*x4 - 6252.92378255669*x6 + 6992.19215508807*x8 - 45.149769199401

dx7 = x8

dx8 = -234.05085207745*U1 + 507.682079801535*U2 - 1662.33394474605*U3 + 3595.91593191811*U4 - 313.337268330785*U5 + 962.833202702448*x10 + 984.476396550774*x2 - 2135.43774816521*x4 + 6992.19215508807*x6 - 15125.3213886305*x8 +

In [80]:
import sympy as sp

# قائمة المتغيرات
x = sp.Matrix(sp.symbols('x1 x2 x3 x4 x5 x6 x7 x8 x9 x10'))
U = sp.Matrix(sp.symbols('U1 U2 U3 U4 U5'))

# dx_ لديك بالفعل بعد التعويض
# لنفترض أن dx_ هو المصفوفة 10x1 الناتجة
# نقوم بفصل A و B بحيث dx_ = A*x + B*U

# فصل المعاملات أمام x
A = dx_.jacobian(x)

# فصل المعاملات أمام U
B = dx_.jacobian(U)

# طباعة النتائج بشكل مصفوفة
sp.init_printing()  # لتنسيق جميل
print("A =")
sp.pprint(A)
print("\nB =")
sp.pprint(B)

# الشكل النهائي للنظام
print("\ndx = A*x + B*U")


A =
⎡0          1          0          0          0          0          0           ↪
⎢                                                                              ↪
⎢0  -457.728705031023  0  192.131965645768   0  -506.422859079951  0  984.4763 ↪
⎢                                                                              ↪
⎢0          0          0          1          0          0          0           ↪
⎢                                                                              ↪
⎢0  192.131965645768   0  -1287.91332802453  0  2400.29060461263   0  -2135.43 ↪
⎢                                                                              ↪
⎢0          0          0          0          0          1          0           ↪
⎢                                                                              ↪
⎢0  -506.422859079951  0  2400.29060461263   0  -6252.92378255669  0  6992.192 ↪
⎢                                                                              ↪
⎢0          0          0